In [0]:
spark.conf.set("fs.azure.account.auth.type.proddatalakefordatabrick.dfs.core.windows.net", "OAuth")
spark.conf.set("fs.azure.account.oauth.provider.type.proddatalakefordatabrick.dfs.core.windows.net", "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
spark.conf.set("fs.azure.account.oauth2.client.id.proddatalakefordatabrick.dfs.core.windows.net", "8b03b30e-c838-4568-a41e-a801634f0e96")
spark.conf.set("fs.azure.account.oauth2.client.secret.proddatalakefordatabrick.dfs.core.windows.net", dbutils.secrets.get(scope="darshanscope", key="app-secret"))
spark.conf.set("fs.azure.account.oauth2.client.endpoint.proddatalakefordatabrick.dfs.core.windows.net", "https://login.microsoftonline.com/5ff33f54-37cd-4d6d-9156-f6c00f164999/oauth2/token")

In [0]:
df=spark.readStream.format('cloudFiles').option('cloudFiles.format','csv').option('cloudFiles.schemaLocation',f'abfss://obtfolder@proddatalakefordatabrick.dfs.core.windows.net/destination/checkpoint_Data').load(f'abfss://obtfolder@proddatalakefordatabrick.dfs.core.windows.net/source/data')

In [0]:

#Cleaning the Headers before loading to bronze schema as Delta doen't accept special charecters
import re
clean_cols = [
    re.sub(r'[^a-zA-Z0-9_]', '_', col.strip()).lower()
    for col in df.columns
]

df = df.toDF(*clean_cols)

df.writeStream.format('delta').outputMode('append').option('header',True).option('checkpointLocation',f'abfss://obtfolder@proddatalakefordatabrick.dfs.core.windows.net/destination/checkpoint_Data').trigger(once=True).toTable("supply_chain_project.bronze.bronze_obt")

In [0]:
#df.writeStream.format('csv').outputMode('append').option('header',True).option('checkpointLocation',f'abfss://obtfolder@proddatalakefordatabrick.dfs.core.windows.net/destination/checkpoint_Data').option('path',f'abfss://obtfolder@proddatalakefordatabrick.dfs.core.windows.net/destination/data').trigger(once=True).start()

In [0]:
#import re

#clean_cols = [
#    re.sub(r'[^a-zA-Z0-9_]', '_', col.strip()).lower()
#    for col in bronze_df.columns
#]

#bronze_df = bronze_df.toDF(*clean_cols)

#bronze_df.write.format("delta").mode("overwrite").saveAsTable("supply_chain_project.bronze.bronze_obt")

In [0]:
%sql
create schema supply_chain_project.bronze


In [0]:
df=spark.read.table("supply_chain_project.bronze.bronze_obt")

In [0]:
df.display()


In [0]:
from pyspark.sql.functions import *


df.select(col("customer_id"),
        col("customer_fname"),
        col("customer_lname"),
        col("customer_country"),
        col("customer_state"),
        col("customer_segment"),
        col("order_date_dateorders")).display()

In [0]:
%sql
select * from supply_chain_project.gold.dim_customer

In [0]:
order=spark.read.table("supply_chain_project.gold.fact_orders")
products=spark.read.table("supply_chain_project.gold.dim_products")
customers=spark.read.table("supply_chain_project.gold.dim_customer")


display(products.count())
